# 02 · Silver layer: clean, standard, checked

**Input:** the Bronze tables in `IAIP\warehouse\bronze\` (raw text, exactly as downloaded).
**Output:** clean tables in `IAIP\warehouse\silver\` plus a **data-quality report** (proposal E3).

What Silver does:
| Problem in Bronze | Silver fix |
|---|---|
| numbers stored as text (`"4,440"`) | real numbers (4440) |
| 3 spellings of the same state ("J & K #", "Jammu and Kashmir") | **one code per state** (`JK`) via a mapping table |
| months as text (`"2025-Jan"`) | real dates (2025-01-01), plus year and month |
| 3,239 raw maker names ("TATA MOTORS LTD", "TATA MOTORS PASSENGER VEHICLES LTD") | cleaned name **+ brand** ("Tata Motors") |
| 26 emission norms, 17 vehicle categories | readable **groups** (BS-VI, BS-IV …; Two Wheeler, Goods …) |
| "Total" rows mixed with data | used for checks, then removed |
| October 2026 is not finished | flagged as **partial** |

Run top to bottom (Shift + Enter). About 3–6 minutes.

## Step 1 · Start Spark

In [1]:
import os, sys, re
from pathlib import Path
from datetime import datetime
from functools import reduce

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, DataFrame, functions as F

spark = (SparkSession.builder
         .appName("IAIP-02-Silver")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.ansi.enabled", "false")   # a bad value becomes empty (null) instead of crashing; we count them in the checks
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

BASE = Path.cwd().parent
WH   = BASE / "warehouse"
assert (WH / "bronze").exists(), "Run 01_bronze first."
RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def bronze(name):
    return spark.read.parquet((WH / "bronze" / name).as_uri())

def save(df, name, partition_by=None):
    w = df.write.mode("overwrite")
    if partition_by:
        w = w.partitionBy(*partition_by)
    w.parquet((WH / "silver" / name).as_uri())
    print(f"saved silver/{name}")

def to_number(col):
    """'4,440' -> 4440 ; '' / 'NA' / '-' -> null"""
    c = F.trim(F.regexp_replace(col, ",", ""))
    return F.when(c.isNull() | c.isin("", "NA", "-"), F.lit(None)).otherwise(c.cast("double"))

QC = []   # the data-quality report: one line per check
def qc(check, dataset, expected, actual, passed, details=""):
    QC.append({"check": check, "dataset": dataset, "expected": str(expected), "actual": str(actual),
               "passed": bool(passed), "details": details, "run_ts": RUN_TS})
    print(("PASS " if passed else "FAIL ") + f"| {check} | {dataset} | expected {expected} | actual {actual} {details}")

print("Spark", spark.version)

c:\Users\Arka\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0


## Step 2 · One agreed name for every state (`dim_state` + mapping table)

Each state gets a **2-letter code** (the same codes VAHAN itself uses). A mapping table records **every spelling** seen in every source and which code it means. The check at the end: **0 unmapped names**.

Notes we keep for the report:
- VAHAN has **"UT of DNH and DD"**; the road-safety file lists "Dadra & Nagar Haveli*" (which already includes Daman & Diu) and an empty "Daman & Diu" row. Both map to `DD`.
- Road-safety **J&K includes Ladakh for 2019–2020**.

In [2]:
# code, standard name, region, is_union_territory, name used by VAHAN
STATES = [
 ("AN","Andaman and Nicobar Islands","East",True,"Andaman & Nicobar Island"),
 ("AP","Andhra Pradesh","South",False,"Andhra Pradesh"),
 ("AR","Arunachal Pradesh","North-East",False,"Arunachal Pradesh"),
 ("AS","Assam","North-East",False,"Assam"),
 ("BR","Bihar","East",False,"Bihar"),
 ("CH","Chandigarh","North",True,"Chandigarh"),
 ("CG","Chhattisgarh","Central",False,"Chhattisgarh"),
 ("DL","Delhi","North",True,"Delhi"),
 ("GA","Goa","West",False,"Goa"),
 ("GJ","Gujarat","West",False,"Gujarat"),
 ("HR","Haryana","North",False,"Haryana"),
 ("HP","Himachal Pradesh","North",False,"Himachal Pradesh"),
 ("JK","Jammu and Kashmir","North",True,"Jammu and Kashmir"),
 ("JH","Jharkhand","East",False,"Jharkhand"),
 ("KA","Karnataka","South",False,"Karnataka"),
 ("KL","Kerala","South",False,"Kerala"),
 ("LA","Ladakh","North",True,"Ladakh"),
 ("LD","Lakshadweep","South",True,"Lakshadweep"),
 ("MP","Madhya Pradesh","Central",False,"Madhya Pradesh"),
 ("MH","Maharashtra","West",False,"Maharashtra"),
 ("MN","Manipur","North-East",False,"Manipur"),
 ("ML","Meghalaya","North-East",False,"Meghalaya"),
 ("MZ","Mizoram","North-East",False,"Mizoram"),
 ("NL","Nagaland","North-East",False,"Nagaland"),
 ("OR","Odisha","East",False,"Odisha"),
 ("PY","Puducherry","South",True,"Puducherry"),
 ("PB","Punjab","North",False,"Punjab"),
 ("RJ","Rajasthan","North",False,"Rajasthan"),
 ("SK","Sikkim","North-East",False,"Sikkim"),
 ("TN","Tamil Nadu","South",False,"Tamil Nadu"),
 ("TG","Telangana","South",False,"Telangana"),
 ("TR","Tripura","North-East",False,"Tripura"),
 ("DD","Dadra and Nagar Haveli and Daman and Diu","West",True,"UT of DNH and DD"),
 ("UP","Uttar Pradesh","North",False,"Uttar Pradesh"),
 ("UK","Uttarakhand","North",False,"Uttarakhand"),
 ("WB","West Bengal","East",False,"West Bengal"),
]
# road-safety spellings that differ from the standard name
ROAD_SPECIAL = {"Andaman & Nicobar Islands": ("AN", ""),
                "Dadra & Nagar Haveli*":     ("DD", "includes Daman & Diu"),
                "Daman & Diu":               ("DD", "empty row; counted in Dadra & Nagar Haveli"),
                "J & K #":                   ("JK", "includes Ladakh for 2019-2020")}

dim_state = spark.createDataFrame([(c, n, r, ut) for c, n, r, ut, _ in STATES],
                                  ["state_code", "state_name", "region", "is_union_territory"])

std_to_code = {n: c for c, n, _, _, _ in STATES}
map_rows = [("VAHAN", v, c, "") for c, _, _, _, v in STATES]
road_names = [r["state_raw"] for r in bronze("road_safety_state_year")
              .filter(F.col("state_raw").isNotNull() & ~F.col("is_total")).select("state_raw").distinct().collect()]
for name in sorted(road_names):
    code_, note = ROAD_SPECIAL.get(name, (std_to_code.get(name), ""))
    map_rows.append(("ROAD_SAFETY", name, code_, note))
state_map = spark.createDataFrame(map_rows, "source string, raw_name string, state_code string, note string")

unmapped = state_map.filter(F.col("state_code").isNull())
qc("state names mapped", "all sources", 0, unmapped.count(), unmapped.count() == 0,
   f"({state_map.count()} spellings -> {dim_state.count()} states)")
unmapped.show(truncate=False)

vahan_map = F.broadcast(state_map.filter(F.col("source") == "VAHAN").select(F.col("raw_name").alias("state_raw"), "state_code"))
road_map  = F.broadcast(state_map.filter(F.col("source") == "ROAD_SAFETY").select(F.col("raw_name").alias("state_raw"), "state_code"))

PASS | state names mapped | all sources | expected 0 | actual 0 (73 spellings -> 36 states)
+------+--------+----------+----+
|source|raw_name|state_code|note|
+------+--------+----------+----+
+------+--------+----------+----+



## Step 3 · Bronze checks before cleaning

**Totals check:** every VAHAN file has its own *Total* row or column. If the states (or makers) add up exactly to VAHAN's own total, nothing was lost or doubled in Bronze.
**Number check:** count values that could not be turned into numbers.
**Duplicate rows:** VAHAN repeats one maker row twice in `maker_NT_2022.csv`; Silver removes exact repeats and reports them.

In [3]:
DEDUP_KEYS = ["source_file", "row_label", "column_label", "value_raw"]

# VAHAN itself repeats one maker row twice in one file (found by the totals check on the first run).
# An exact repeat adds no information, so Silver removes it and records it here.
mk = bronze("reg_maker_state_year")
dups = (mk.groupBy(DEDUP_KEYS).count().filter("count > 1")
          .select("source_file", F.regexp_replace("row_label", r"\s+", " ").alias("maker")).distinct().collect())
qc("exact duplicate rows in source removed", "reg_maker_state_year", "report", len(dups), True,
   "; ".join(f"{r['source_file']}: {r['maker'].strip()}" for r in dups))

def totals_check(name, total_is_row=True):
    b = bronze(name).dropDuplicates(DEDUP_KEYS).withColumn("v", to_number(F.col("value_raw")))
    if total_is_row:      # states are rows; 'Total' is a row
        keys = ["source_file", "column_label"]
        parts = b.filter(F.col("row_label") != "Total").groupBy(keys).agg(F.sum("v").alias("s"))
        tot   = b.filter(F.col("row_label") == "Total").select(*keys, F.col("v").alias("t"))
    else:                 # maker files: makers are rows; 'Total' is a column
        keys = ["source_file", "row_label"]
        parts = b.filter(F.col("column_label") != "Total").groupBy(keys).agg(F.sum("v").alias("s"))
        tot   = b.filter(F.col("column_label") == "Total").select(*keys, F.col("v").alias("t"))
    j = parts.join(tot, keys, "full")
    n, bad = j.count(), j.filter(F.coalesce("s", F.lit(0.0)) != F.coalesce("t", F.lit(0.0))).count()
    qc("parts add up to VAHAN's own Total", name, 0, bad, bad == 0, f"({n:,} totals compared)")

    not_numeric = b.filter(F.col("value_raw").isNotNull() & ~F.trim("value_raw").isin("", "NA", "-") & F.col("v").isNull()).count()
    negative = b.filter(F.col("v") < 0).count()
    qc("values converted to numbers", name, 0, not_numeric, not_numeric == 0, "(text that is not a number)")
    qc("no negative values", name, 0, negative, negative == 0)

totals_check("reg_state_month")
totals_check("reg_state_category_year")
totals_check("reg_state_norm_year")
totals_check("reg_maker_state_year", total_is_row=False)

PASS | exact duplicate rows in source removed | reg_maker_state_year | expected report | actual 1 maker_NT_2022.csv: SIDDHAKALA ENGINEERING AND WELDING WORKS
PASS | parts add up to VAHAN's own Total | reg_state_month | expected 0 | actual 0 (1,204 totals compared)
PASS | values converted to numbers | reg_state_month | expected 0 | actual 0 (text that is not a number)
PASS | no negative values | reg_state_month | expected 0 | actual 0 
PASS | parts add up to VAHAN's own Total | reg_state_category_year | expected 0 | actual 0 (252 totals compared)
PASS | values converted to numbers | reg_state_category_year | expected 0 | actual 0 (text that is not a number)
PASS | no negative values | reg_state_category_year | expected 0 | actual 0 
PASS | parts add up to VAHAN's own Total | reg_state_norm_year | expected 0 | actual 0 (378 totals compared)
PASS | values converted to numbers | reg_state_norm_year | expected 0 | actual 0 (text that is not a number)
PASS | no negative values | reg_state_no

## Step 4 · Silver `reg_state_month`: registrations by state × month × fuel group × vehicle type

- text month → **date**; `year` and `month` columns
- **is_partial_month**: the month you downloaded in (Oct 2026) is not finished yet
- **Fuel completeness check:** for every state, month and type, **EV + PETROL + DIESEL + HYBRID + GAS + OTHER must equal ALL**. After the check we keep the 6 groups (ALL would double-count) and add `is_ev`.

In [4]:
manifest = F.broadcast(bronze("_file_manifest")
                       .select("source_file", F.to_date(F.substring("file_modified", 1, 10)).alias("downloaded_on")))

sm_all = (bronze("reg_state_month")
          .filter(~F.col("is_total"))
          .withColumnRenamed("row_label", "state_raw")
          .join(vahan_map, "state_raw", "left")
          .withColumn("month_date", F.to_date(F.concat(F.col("column_label"), F.lit("-01")), "yyyy-MMM-dd"))
          .withColumn("registrations", to_number(F.col("value_raw")).cast("long"))
          .join(manifest, "source_file", "left")
          .withColumn("year", F.year("month_date"))
          .withColumn("month", F.month("month_date"))
          .withColumn("is_partial_month", F.col("month_date") >= F.trunc("downloaded_on", "month"))
          .select("state_code", "month_date", "year", "month", "vehicle_type", "fuel_group",
                  "registrations", "is_partial_month", "source_file", "downloaded_on", "load_ts"))

bad_dates = sm_all.filter(F.col("month_date").isNull()).count()
qc("month labels converted to dates", "reg_state_month", 0, bad_dates, bad_dates == 0)

# completeness: the 6 fuel groups must add up to ALL, cell by cell
keys = ["state_code", "month_date", "vehicle_type"]
all_f = sm_all.filter(F.col("fuel_group") == "ALL").select(*keys, F.col("registrations").alias("all_reg"))
grp_f = sm_all.filter(F.col("fuel_group") != "ALL").groupBy(keys).agg(F.sum("registrations").alias("sum_groups"),
                                                                       F.countDistinct("fuel_group").alias("n_groups"))
cmp = all_f.join(grp_f, keys, "full")
cells = cmp.count()
mism = cmp.filter((F.coalesce("all_reg", F.lit(0)) != F.coalesce("sum_groups", F.lit(0))) | (F.col("n_groups") != 6)).count()
qc("6 fuel groups add up to ALL", "reg_state_month", 0, mism, mism == 0, f"({cells:,} state-month-type cells)")

FUEL_DIM = [("EV", True, "Battery electric (ELECTRIC(BOV), PURE EV)"),
            ("PETROL", False, "Petrol incl. E20"),
            ("DIESEL", False, "Diesel"),
            ("HYBRID", False, "Strong / plug-in / mild hybrids"),
            ("GAS", False, "CNG, LPG, LNG incl. bi-fuel"),
            ("OTHER", False, "Ethanol, methanol, hydrogen, solar, not applicable")]
dim_fuel = spark.createDataFrame(FUEL_DIM, "fuel_group string, is_ev boolean, description string")

reg_state_month = sm_all.filter(F.col("fuel_group") != "ALL").join(F.broadcast(dim_fuel.select("fuel_group", "is_ev")), "fuel_group")

# no gaps: every state x type x fuel group has every month from Jan 2020 to the last month
n_months = reg_state_month.select("month_date").distinct().count()
gaps = (reg_state_month.groupBy("state_code", "vehicle_type", "fuel_group")
        .agg(F.countDistinct("month_date").alias("m")).filter(F.col("m") != n_months).count())
qc("no missing months", "reg_state_month", 0, gaps, gaps == 0, f"({n_months} months each)")

reg_state_month.groupBy("year").agg(F.sum("registrations").alias("registrations")).orderBy("year").show()

PASS | month labels converted to dates | reg_state_month | expected 0 | actual 0 
PASS | 6 fuel groups add up to ALL | reg_state_month | expected 0 | actual 0 (5,976 state-month-type cells)
PASS | no missing months | reg_state_month | expected 0 | actual 0 (83 months each)
+----+-------------+
|year|registrations|
+----+-------------+
|2020|     19609105|
|2021|     19779244|
|2022|     22485053|
|2023|     24960957|
|2024|     27236092|
|2025|     29290090|
|2026|     28427696|
+----+-------------+



## Step 5 · Silver yearly tables: vehicle category, emission norm, maker

- **Vehicle category** (17) → `category_group`: Two Wheeler, Three Wheeler, Car / LMV, Passenger (bus/taxi), Goods, Other
- **Emission norm** (26) → `norm_group`: BS-VI, BS-IV, BS-III or older, EURO (imported), Tractor/Construction (TREM/CEV), Not applicable
- **Maker**: clean the name (spaces, line breaks, upper case) and assign a **brand** using simple rules; rows with 0 registrations are dropped (a maker that sold nothing in a state)
- `is_partial_year`: 2026 is not finished

In [5]:
CATEGORY_GROUP = {
 "TWO WHEELER(NT)": "Two Wheeler", "TWO WHEELER(T)": "Two Wheeler", "TWO WHEELER (Invalid Carriage)": "Two Wheeler",
 "THREE WHEELER(NT)": "Three Wheeler", "THREE WHEELER(T)": "Three Wheeler", "THREE WHEELER (Invalid Carriage)": "Three Wheeler",
 "LIGHT MOTOR VEHICLE": "Car / LMV", "FOUR WHEELER (Invalid Carriage)": "Car / LMV",
 "LIGHT PASSENGER VEHICLE": "Passenger (bus/taxi)", "MEDIUM PASSENGER VEHICLE": "Passenger (bus/taxi)", "HEAVY PASSENGER VEHICLE": "Passenger (bus/taxi)",
 "LIGHT GOODS VEHICLE": "Goods", "MEDIUM GOODS VEHICLE": "Goods", "HEAVY GOODS VEHICLE": "Goods",
 "MEDIUM MOTOR VEHICLE": "Other", "HEAVY MOTOR VEHICLE": "Other", "OTHER THAN MENTIONED ABOVE": "Other"}

def norm_group(n):
    u = n.upper()
    if "TREM" in u or "CEV" in u:              return "Tractor/Construction (TREM/CEV)"
    if u == "BHARAT STAGE VI":                 return "BS-VI"
    if u in ("BHARAT STAGE IV", "BHARAT STAGE III/IV"): return "BS-IV"
    if u.startswith("BHARAT STAGE"):           return "BS-III or older"
    if u.startswith("EURO"):                   return "EURO (imported)"
    return "Not applicable / not available"

def year_from_file():
    return F.regexp_extract("source_file", r"_(\d{4})\.csv$", 1).cast("int")

def yearly(name, label_col):
    return (bronze(name).filter(~F.col("is_total"))
            .withColumnRenamed("row_label", "state_raw").join(vahan_map, "state_raw", "left")
            .withColumn("year", year_from_file())
            .withColumn("registrations", to_number(F.col("value_raw")).cast("long"))
            .withColumn("is_partial_year", F.col("year") == 2026)
            .withColumnRenamed("column_label", label_col))

cat_map = F.broadcast(spark.createDataFrame(list(CATEGORY_GROUP.items()), "vehicle_category string, category_group string"))
reg_state_category_year = (yearly("reg_state_category_year", "vehicle_category").join(cat_map, "vehicle_category", "left")
    .select("state_code", "year", "vehicle_type", "vehicle_category", "category_group", "registrations", "is_partial_year", "source_file", "load_ts"))

norms = [r[0] for r in bronze("reg_state_norm_year").select("column_label").distinct().collect() if r[0] != "Total"]
norm_map = F.broadcast(spark.createDataFrame([(n, norm_group(n)) for n in norms], "emission_norm string, norm_group string"))
reg_state_norm_year = (yearly("reg_state_norm_year", "emission_norm").join(norm_map, "emission_norm", "left")
    .select("state_code", "year", "vehicle_type", "emission_norm", "norm_group", "registrations", "is_partial_year", "source_file", "load_ts"))

qc("every vehicle category has a group", "reg_state_category_year", 0,
   reg_state_category_year.filter(F.col("category_group").isNull()).count(),
   reg_state_category_year.filter(F.col("category_group").isNull()).count() == 0)
norm_map.orderBy("norm_group", "emission_norm").show(30, truncate=False)

PASS | every vehicle category has a group | reg_state_category_year | expected 0 | actual 0 
+-----------------------+-------------------------------+
|emission_norm          |norm_group                     |
+-----------------------+-------------------------------+
|BHARAT STAGE I         |BS-III or older                |
|BHARAT STAGE II        |BS-III or older                |
|BHARAT STAGE III       |BS-III or older                |
|BHARAT STAGE III/IV    |BS-IV                          |
|BHARAT STAGE IV        |BS-IV                          |
|BHARAT STAGE VI        |BS-VI                          |
|EURO 1                 |EURO (imported)                |
|EURO 2                 |EURO (imported)                |
|EURO 3                 |EURO (imported)                |
|EURO 4                 |EURO (imported)                |
|EURO 6                 |EURO (imported)                |
|EURO 6A                |EURO (imported)                |
|EURO 6AD               |EURO (import

In [6]:
# Maker brand rules: first matching pattern wins (Java regular expressions)
BRAND_RULES = [
 ("^HERO ELECTRIC", "Hero Electric"), ("^HERO MOTOCORP", "Hero MotoCorp"), ("^HONDA", "Honda"),
 ("^TVS ", "TVS Motor"), ("^BAJAJ", "Bajaj Auto"), ("^MARUTI", "Maruti Suzuki"), ("^SUZUKI", "Suzuki Motorcycle"),
 ("ROYAL.ENFIELD", "Royal Enfield"), ("^SML ", "SML Isuzu"), ("^MAHINDRA(?! & ASSOCIATES)", "Mahindra"),
 ("YAMAHA", "Yamaha"), ("^HYUNDAI", "Hyundai"), ("^TATA (MOTORS|PASSENGER)", "Tata Motors"), ("^KIA ", "Kia"),
 ("^TOYOTA", "Toyota"), ("^OLA ELECTRIC", "Ola Electric"), ("^ASHOK LEYLAND", "Ashok Leyland"), ("^PIAGGIO", "Piaggio"),
 ("^ATHER", "Ather Energy"), ("^INTERNATIONAL TRACTORS", "Sonalika (ITL)"), ("^TAFE|^EICHER TRACTORS", "TAFE"),
 ("^ESCORTS|^KUBOTA", "Escorts Kubota"), ("SKODA|VOLKSWAGEN", "Skoda-Volkswagen"), ("^VE COMMERCIAL", "VE Commercial (Volvo-Eicher)"),
 ("JOHN DEERE", "John Deere"), ("^RENAULT", "Renault"), ("MG MOTOR", "JSW MG"), ("^CLASSIC LEGENDS", "Classic Legends"),
 ("^CNH ", "CNH Industrial"), ("^JCB ", "JCB"), ("^YC ELECTRIC", "YC Electric"), ("^GREAVES|^AMPERE", "Greaves (Ampere)"),
 ("^NISSAN", "Nissan"), ("^OKINAWA", "Okinawa"), ("^FORCE MOTORS", "Force Motors"), ("^ATUL AUTO", "Atul Auto"),
 ("^BMW ", "BMW"), ("^SAERA", "Saera Electric"), ("^DAIMLER", "Daimler (BharatBenz)"), ("^MERCEDES", "Mercedes-Benz"),
 ("^DILLI ELECTRIC", "Dilli Electric"), ("^BGAUSS", "BGauss"), ("^FORD ", "Ford"), ("^STELLANTIS", "Stellantis"),
 ("^MINI METRO", "Mini Metro EV"), (r"^J\. ?S\. AUTO", "J.S. Auto"), ("^PUR ENERGY", "PURE EV"), ("^RIVER ", "River"),
 ("^REVOLT", "Revolt"), ("^KINETIC", "Kinetic"), ("^ACTION CONSTRUCTION", "ACE"), ("^LECTRIX", "Lectrix"),
 ("^TERRA MOTORS", "Terra Motors"), ("^OTHERS$", "Others (as reported by VAHAN)"), ("^LOCAL TRAILER", "Local trailer makers"),
]
LONG_TAIL = "Other makers (long tail)"

maker_clean = F.upper(F.trim(F.regexp_replace(F.col("maker_raw"), r"\s+", " ")))
brand = F.lit(LONG_TAIL)
for pattern, name in reversed(BRAND_RULES):          # build: when(rule1).when(rule2)...otherwise(long tail)
    brand = F.when(F.col("maker_clean").rlike(pattern), F.lit(name)).otherwise(brand)

reg_maker_state_year = (bronze("reg_maker_state_year").dropDuplicates(DEDUP_KEYS).filter(~F.col("is_total"))
    .withColumnRenamed("row_label", "maker_raw").withColumnRenamed("column_label", "state_raw")
    .join(vahan_map, "state_raw", "left")
    .withColumn("maker_clean", maker_clean)
    .withColumn("brand", brand)
    .withColumn("year", year_from_file())
    .withColumn("registrations", to_number(F.col("value_raw")).cast("long"))
    .withColumn("is_partial_year", F.col("year") == 2026)
    .filter(F.col("registrations") > 0)
    .select("maker_clean", "brand", "maker_raw", "state_code", "year", "vehicle_type", "registrations", "is_partial_year", "source_file", "load_ts"))

by_brand = reg_maker_state_year.groupBy("brand").agg(F.sum("registrations").alias("reg"))
total = by_brand.agg(F.sum("reg")).first()[0]
named = by_brand.filter(~F.col("brand").isin(LONG_TAIL, "Others (as reported by VAHAN)", "Local trailer makers")).agg(F.sum("reg")).first()[0]
n_raw = reg_maker_state_year.select("maker_raw").distinct().count()
n_clean = reg_maker_state_year.select("maker_clean").distinct().count()
qc("maker names cleaned", "reg_maker_state_year", "100%", "100%", True, f"({n_raw:,} raw spellings -> {n_clean:,} clean names)")
qc("registrations assigned to a named brand", "reg_maker_state_year", ">= 95%", f"{100*named/total:.2f}%", named/total >= 0.95,
   f"({len(BRAND_RULES)} brand rules)")
by_brand.orderBy(F.desc("reg")).show(15, truncate=False)

PASS | maker names cleaned | reg_maker_state_year | expected 100% | actual 100% (3,245 raw spellings -> 3,245 clean names)
PASS | registrations assigned to a named brand | reg_maker_state_year | expected >= 95% | actual 97.27% (55 brand rules)
+-----------------------------+--------+
|brand                        |reg     |
+-----------------------------+--------+
|Hero MotoCorp                |38030468|
|Honda                        |31415634|
|TVS Motor                    |21168194|
|Bajaj Auto                   |16500136|
|Maruti Suzuki                |11558341|
|Mahindra                     |7471507 |
|Suzuki Motorcycle            |5790662 |
|Tata Motors                  |5726879 |
|Royal Enfield                |5542679 |
|Yamaha                       |4227472 |
|Hyundai                      |3810475 |
|Other makers (long tail)     |3169385 |
|Kia                          |1597868 |
|Toyota                       |1490479 |
|Others (as reported by VAHAN)|1274570 |
+-----------------

## Step 6 · Do the four tables agree with each other?

Each table was downloaded separately, so we compare **yearly totals per state and type**: fuel vs category vs norm vs maker. 2020–2025 must agree within **0.1%**. 2026 is reported but not judged, because the tables were downloaded on different days (VAHAN adds new October registrations every day).

**Official check:** India's yearly totals compared with the figures VAHAN publishes on its dashboard (2022–2025); target **within 1%**.

In [7]:
k = ["state_code", "year", "vehicle_type"]
ann = {"fuel": reg_state_month.groupBy(k).agg(F.sum("registrations").alias("fuel"))}
ann["category"] = reg_state_category_year.groupBy(k).agg(F.sum("registrations").alias("category"))
ann["norm"]     = reg_state_norm_year.groupBy(k).agg(F.sum("registrations").alias("norm"))
ann["maker"]    = reg_maker_state_year.groupBy(k).agg(F.sum("registrations").alias("maker"))
cross = reduce(lambda a, b: a.join(b, k, "full"), ann.values()).na.fill(0)

national = cross.groupBy("year").agg(*[F.sum(c).alias(c) for c in ann]).orderBy("year")
for other in ["category", "norm", "maker"]:
    national = national.withColumn(f"{other}_vs_fuel_%", F.round(100 * (F.col(other) - F.col("fuel")) / F.col("fuel"), 4))
national.show(truncate=False)

for other in ["category", "norm", "maker"]:
    worst = national.filter(F.col("year") <= 2025).agg(F.max(F.abs(F.col(f"{other}_vs_fuel_%")))).first()[0]
    qc(f"{other} table agrees with fuel table (2020-2025)", "cross-table", "<= 0.1%", f"{worst:.4f}%", worst <= 0.1)

# VAHAN dashboard published yearly totals (from the dashboard home page)
PUBLISHED = {2022: 22486465, 2023: 24962340, 2024: 27237701, 2025: 29291584}
for y, pub in PUBLISHED.items():
    ours = national.filter(F.col("year") == y).first()["fuel"]
    diff = 100 * (ours - pub) / pub
    qc(f"India total {y} vs VAHAN published", "reg_state_month", "within 1%", f"{diff:+.4f}%", abs(diff) <= 1,
       f"(ours {ours:,} vs published {pub:,})")

+----+--------+--------+--------+--------+------------------+--------------+---------------+
|year|fuel    |category|norm    |maker   |category_vs_fuel_%|norm_vs_fuel_%|maker_vs_fuel_%|
+----+--------+--------+--------+--------+------------------+--------------+---------------+
|2020|19609105|19608766|19609164|19609164|-0.0017           |3.0E-4        |3.0E-4         |
|2021|19779244|19778879|19779315|19779315|-0.0018           |4.0E-4        |4.0E-4         |
|2022|22485053|22484720|22485175|22485175|-0.0015           |5.0E-4        |5.0E-4         |
|2023|24960957|24960644|24961071|24961071|-0.0013           |5.0E-4        |5.0E-4         |
|2024|27236092|27235967|27236173|27236173|-5.0E-4           |3.0E-4        |3.0E-4         |
|2025|29290090|29290036|29290142|29290142|-2.0E-4           |2.0E-4        |2.0E-4         |
|2026|28427696|28427696|28427696|28427696|0.0               |0.0           |0.0            |
+----+--------+--------+--------+--------+------------------+---------

## Step 7 · Silver road safety

- accidents and deaths as **numbers**, one row per **state × year**
- state names mapped to codes (Dadra & Nagar Haveli + Daman & Diu → `DD`)
- `jk_includes_ladakh` = true for J&K in 2019–2020
- check: the states add up to the **All India** row

In [8]:
rb = (bronze("road_safety_state_year")
      .withColumn("v", to_number(F.col("value_raw")))
      .withColumn("year", F.col("year").cast("int")))

for measure in ["accidents", "killed"]:
    st = rb.filter(~F.col("is_total") & F.col("state_raw").isNotNull() & (F.col("measure") == measure)).groupBy("year").agg(F.sum("v").alias("s"))
    tt = rb.filter(F.col("is_total") & (F.col("measure") == measure)).select("year", F.col("v").alias("t"))
    bad = st.join(tt, "year", "full").filter(F.coalesce("s", F.lit(0.0)) != F.coalesce("t", F.lit(0.0))).count()
    qc(f"states add up to All India ({measure})", "road_safety_state_year", 0, bad, bad == 0, "(2019-2023)")

road_safety_state_year = (rb.filter(~F.col("is_total") & F.col("state_raw").isNotNull())
    .join(road_map, "state_raw", "left")
    .groupBy("state_code", "year").pivot("measure", ["accidents", "killed"]).agg(F.sum("v"))
    .withColumn("accidents", F.col("accidents").cast("long"))
    .withColumn("killed", F.col("killed").cast("long"))
    .withColumn("jk_includes_ladakh", (F.col("state_code") == "JK") & F.col("year").isin(2019, 2020))
    .filter(F.col("accidents").isNotNull() | F.col("killed").isNotNull()))

nat_raw = bronze("road_safety_national_year")
num_cols = [c for c in nat_raw.columns if c not in ("id", "year", "source_file", "load_ts")]
road_safety_national_year = (nat_raw.filter(F.col("year").rlike(r"^\d{4}$"))
    .select(F.col("year").cast("int").alias("year"), *[to_number(F.col(c)).alias(c) for c in num_cols]))

road_safety_state_year.filter(F.col("state_code").isin("KL", "JK", "LA", "DD")).orderBy("state_code", "year").show(30)

PASS | states add up to All India (accidents) | road_safety_state_year | expected 0 | actual 0 (2019-2023)
PASS | states add up to All India (killed) | road_safety_state_year | expected 0 | actual 0 (2019-2023)
+----------+----+---------+------+------------------+
|state_code|year|accidents|killed|jk_includes_ladakh|
+----------+----+---------+------+------------------+
|        DD|2019|      137|    77|             false|
|        DD|2020|      100|    64|             false|
|        DD|2021|      140|    76|             false|
|        DD|2022|      196|    90|             false|
|        DD|2023|      182|    74|             false|
|        JK|2019|     5796|   996|              true|
|        JK|2020|     4860|   728|              true|
|        JK|2021|     5452|   774|             false|
|        JK|2022|     6092|   805|             false|
|        JK|2023|     6298|   893|             false|
|        KL|2019|    41111|  4440|             false|
|        KL|2020|    27877|  2979

## Step 8 · No duplicates, then save Silver

Each table must have **exactly one row per key** (e.g. one value per state × month × type × fuel group). If the same month were loaded twice, this check would catch it.

In [9]:
SILVER = {
 "reg_state_month":         (reg_state_month,         ["state_code", "month_date", "vehicle_type", "fuel_group"], ["year"]),
 "reg_state_category_year": (reg_state_category_year, ["state_code", "year", "vehicle_type", "vehicle_category"], ["year"]),
 "reg_state_norm_year":     (reg_state_norm_year,     ["state_code", "year", "vehicle_type", "emission_norm"],    ["year"]),
 "reg_maker_state_year":    (reg_maker_state_year,    ["maker_raw", "state_code", "year", "vehicle_type"],        ["year"]),
 "road_safety_state_year":  (road_safety_state_year,  ["state_code", "year"], None),
 "road_safety_national_year": (road_safety_national_year, ["year"], None),
}
for name, (df, key, _) in SILVER.items():
    dup = df.groupBy(key).count().filter(F.col("count") > 1).count()
    qc("no duplicate keys", name, 0, dup, dup == 0, f"(key: {', '.join(key)})")
    nostate = df.filter(F.col("state_code").isNull()).count() if "state_code" in df.columns else 0
    qc("every row has a state code", name, 0, nostate, nostate == 0)

for name, (df, _, part) in SILVER.items():
    save(df, name, part)
save(dim_state, "dim_state")
save(state_map, "state_name_map")
save(dim_fuel, "dim_fuel")

PASS | no duplicate keys | reg_state_month | expected 0 | actual 0 (key: state_code, month_date, vehicle_type, fuel_group)
PASS | every row has a state code | reg_state_month | expected 0 | actual 0 
PASS | no duplicate keys | reg_state_category_year | expected 0 | actual 0 (key: state_code, year, vehicle_type, vehicle_category)
PASS | every row has a state code | reg_state_category_year | expected 0 | actual 0 
PASS | no duplicate keys | reg_state_norm_year | expected 0 | actual 0 (key: state_code, year, vehicle_type, emission_norm)
PASS | every row has a state code | reg_state_norm_year | expected 0 | actual 0 
PASS | no duplicate keys | reg_maker_state_year | expected 0 | actual 0 (key: maker_raw, state_code, year, vehicle_type)
PASS | every row has a state code | reg_maker_state_year | expected 0 | actual 0 
PASS | no duplicate keys | road_safety_state_year | expected 0 | actual 0 (key: state_code, year)
PASS | every row has a state code | road_safety_state_year | expected 0 | actu

## Step 9 · The data-quality report (E3)

Saved as Parquet **and** as a CSV you can open in Excel: `IAIP\warehouse\silver\_quality_report_csv\`.

In [10]:
report = spark.createDataFrame(QC)
save(report, "_quality_report")
(report.coalesce(1).write.mode("overwrite").option("header", True)
       .csv((WH / "silver" / "_quality_report_csv").as_uri()))

passed = report.filter("passed").count()
print(f"\nDATA QUALITY: {passed} of {report.count()} checks passed\n")
report.select("passed", "check", "dataset", "expected", "actual").show(60, truncate=False)

saved silver/_quality_report

DATA QUALITY: 41 of 41 checks passed

+------+-------------------------------------------------+-------------------------+---------+--------+
|passed|check                                            |dataset                  |expected |actual  |
+------+-------------------------------------------------+-------------------------+---------+--------+
|true  |state names mapped                               |all sources              |0        |0       |
|true  |exact duplicate rows in source removed           |reg_maker_state_year     |report   |1       |
|true  |parts add up to VAHAN's own Total                |reg_state_month          |0        |0       |
|true  |values converted to numbers                      |reg_state_month          |0        |0       |
|true  |no negative values                               |reg_state_month          |0        |0       |
|true  |parts add up to VAHAN's own Total                |reg_state_category_year  |0        |0     

In [11]:
# Read back every Silver table
for name in list(SILVER) + ["dim_state", "state_name_map", "dim_fuel", "_quality_report"]:
    df = spark.read.parquet((WH / "silver" / name).as_uri())
    print(f"{name:28} {df.count():>9,} rows   {', '.join(df.columns)[:100]}")

reg_state_month                 35,856 rows   fuel_group, state_code, month_date, month, vehicle_type, registrations, is_partial_month, source_fil
reg_state_category_year          8,568 rows   state_code, vehicle_type, vehicle_category, category_group, registrations, is_partial_year, source_f
reg_state_norm_year             13,104 rows   state_code, vehicle_type, emission_norm, norm_group, registrations, is_partial_year, source_file, lo
reg_maker_state_year            64,082 rows   maker_clean, brand, maker_raw, state_code, vehicle_type, registrations, is_partial_year, source_file
road_safety_state_year             178 rows   state_code, year, accidents, killed, jk_includes_ladakh
road_safety_national_year           12 rows   year, road_accidents_000, road_accident_deaths_000, road_accident_injuries_000, registered_vehicles_
dim_state                           36 rows   state_code, state_name, region, is_union_territory
state_name_map                      73 rows   source, raw_name, st

In [12]:
# A first insight: EV share of new registrations, India, by year (Non-Transport vs Transport)
s = spark.read.parquet((WH / "silver" / "reg_state_month").as_uri())
(s.groupBy("year", "vehicle_type")
   .agg((100 * F.sum(F.when(F.col("is_ev"), F.col("registrations")).otherwise(0)) / F.sum("registrations")).alias("ev_share_%"))
   .groupBy("year").pivot("vehicle_type", ["NT", "T"]).agg(F.round(F.first("ev_share_%"), 2))
   .orderBy("year").show())

+----+----+-----+
|year|  NT|    T|
+----+----+-----+
|2020|0.17| 7.44|
|2021|0.95|12.83|
|2022|3.37| 19.2|
|2023|4.36|23.96|
|2024|5.31|26.76|
|2025|5.76|28.85|
|2026|8.44|29.96|
+----+----+-----+



## Done ✅

Next notebook: **03 · Gold**: the star schema (`fact_registration`, `fact_road_safety`, dimensions) and the summary views from your proposal (`ev_share_state_month`, `fuel_mix_trend`, `state_safety_profile` …) in Spark SQL, exported for Power BI.

In [ ]:
spark.stop()